In [1]:
!git clone https://github.com/HungKingiscoming/road-extraction.git

Cloning into 'road-extraction'...
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 593 (delta 41), reused 41 (delta 18), pack-reused 521 (from 1)
Receiving objects: 100% (593/593), 7.08 MiB | 25.08 MiB/s, done.
Resolving deltas: 100% (318/318), done.


In [2]:
!pip install -r /kaggle/working/road-extraction/requirements.txt

In [3]:
!pip install ptflops

In [4]:
cd /kaggle/working/road-extraction/

/kaggle/working/road-extraction


In [5]:
from pathlib import Path
import zipfile
import torch

# Thư mục .pt đã bị Kaggle giải nén
src = Path(
    "/kaggle/input/datasets/datnguyentien204/road-8-27-2026/best_calibrated_road_iou.pt"
)

# File weight được khôi phục
dst = Path("/kaggle/working/best_fixed_iou.pt")

assert src.is_dir(), f"Không tìm thấy thư mục đã giải nén: {src}"

# Tìm thư mục chứa data.pkl của checkpoint PyTorch
data_pkls = list(src.rglob("data.pkl"))
assert len(data_pkls) == 1, f"Tìm thấy {len(data_pkls)} file data.pkl"

checkpoint_root = data_pkls[0].parent

# Khôi phục đúng cấu trúc ZIP serialization của PyTorch
with zipfile.ZipFile(
    dst,
    mode="w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as archive:
    for file in checkpoint_root.rglob("*"):
        if file.is_file():
            relative = file.relative_to(checkpoint_root)
            archive.write(file, arcname=str(Path("archive") / relative))

print("Đã tạo:", dst)
print(f"Dung lượng: {dst.stat().st_size / 1024**2:.1f} MB")

# Kiểm tra file weight
checkpoint = torch.load(dst, map_location="cpu", weights_only=False)
print("Load thành công")
print("Các key:", checkpoint.keys())

Đã tạo: /kaggle/working/best_fixed_iou.pt
Dung lượng: 344.7 MB
Load thành công
Các key: dict_keys(['epoch', 'model', 'ema', 'ema_updates', 'optimizer', 'scheduler', 'scaler', 'best_fixed_road_iou', 'best_calibrated_road_iou', 'validation', 'args'])


In [ ]:
%cd /kaggle/working/road-extraction

!OMP_NUM_THREADS=2 TORCH_NCCL_ASYNC_ERROR_HANDLING=1 \
torchrun --standalone --nproc_per_node=2 \
  train.py \
  --dataset massachusetts \
  --data_root /kaggle/input/datasets/datnguyentien204/massachu/massachusets \
  --train_image_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/images \
  --train_mask_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/labels \
  --train_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/train.txt \
  --test_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/test.txt \
  --crop_size 1024 \
  --batch_size 8 \
  --accumulation_steps 1 \
  --epochs 100 \
  --lr 5e-5 \
  --dual_branch_lr_factor 0.75 \
  --backbone_lr_factor 0.20 \
  --early_encoder_lr_factor 0.10 \
  --weight_decay 1e-4 \
  --warmup_epochs 3 \
  --warmup_start_factor 0.10 \
  --min_lr_ratio 0.0 \
  --grad_clip 5.0 \
  --ema_decay 0.999 \
  --detail_channels 96 \
  --semantic_channels 192 \
  --dappm_channels 32 \
  --dappm_pool_sizes 1 2 4 8 \
  --detail_blocks 2 2 \
  --semantic_blocks 2 \
  --fusion_blocks 1 \
  --bilateral_fusion spatial \
  --decoder_s4_channels 64 \
  --decoder_s2_channels 32 \
  --full_channels 24 \
  --dropout 0.05 \
  --aux_weight 0.15 \
  --aux_start_epoch 0 \
  --aux_warmup_epochs 0 \
  --skeleton_iterations 8 \
  --road_crop_probability 0.0 \
  --road_occlusion_probability 0.0 \
  --progressive_unfreeze \
  --unfreeze_dual_branch_epoch 0 \
  --unfreeze_layer3_epoch 5 \
  --unfreeze_layer2_epoch 10 \
  --unfreeze_all_epoch 15 \
  --freeze_encoder_bn \
  --num_workers 4 \
  --prefetch_factor 2 \
  --use_amp \
  --channels_last \
  --pretrained_checkpoint /kaggle/working/best_fixed_iou.pt \
  --transfer_weights ema \
  --save_dir ./checkpoints/roadfusion_spatial_bilateral_ft40

/kaggle/working/road-extraction
[W914 08:08:58.217719419 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank 0] Initializing NCCL on cuda:0 (world_size=2)...
[W914 08:09:00.723594075 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank 1] Initializing NCCL on cuda:1 (world_size=2)...
[W914 08:09:00.726913565 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank 1] NCCL process group ready
[rank 0] NCCL process group ready
[startup 1/5] DDP initialized: world_size=2, device=cuda:0
[startup 2/5] Resolving image/mask pairs and DataLoaders...
TXT split: train=600, val=61, test=117 | val=first 61 entries of /kaggle/input/datasets/datnguyentien204/massachu/massachusets/test.txt
[startup 3/5] Scanning 600 training masks to estimate the road class weight...
[startup] class-weight scan 1/600 masks
[startup] class-weight scan 60/600 masks
[startup] class-weight scan 120/600 masks
[st

In [ ]:
!python test_native.py \
  --ckpt ./checkpoints/roadfusion_spatial_bilateral_ft40/best_calibrated_road_iou.pt \
  --dataset massachusetts --subset test178 --thr 0.58 --weights ema \
  --tta-mode flip4 --tta-merge probabilities --amp float16


In [ ]:
%cd /kaggle/working/road-extraction

!OMP_NUM_THREADS=2 TORCH_NCCL_ASYNC_ERROR_HANDLING=1 \
torchrun --standalone --nproc_per_node=2 \
  train.py \
  --dataset massachusetts \
  --data_root /kaggle/input/datasets/datnguyentien204/massachu/massachusets \
  --train_image_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/images \
  --train_mask_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/labels \
  --train_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/train.txt \
  --test_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/test.txt \
  --crop_size 1024 \
  --batch_size 8 \
  --accumulation_steps 1 \
  --epochs 100 \
  --lr 5e-5 \
  --dual_branch_lr_factor 0.75 \
  --backbone_lr_factor 0.20 \
  --early_encoder_lr_factor 0.10 \
  --weight_decay 1e-4 \
  --warmup_epochs 3 \
  --warmup_start_factor 0.10 \
  --min_lr_ratio 0.0 \
  --grad_clip 5.0 \
  --ema_decay 0.999 \
  --detail_channels 96 \
  --semantic_channels 192 \
  --dappm_channels 32 \
  --dappm_pool_sizes 1 2 4 8 \
  --detail_blocks 2 2 \
  --semantic_blocks 2 \
  --fusion_blocks 1 \
  --bilateral_fusion spatial \
  --decoder_s4_channels 64 \
  --decoder_s2_channels 32 \
  --full_channels 24 \
  --dropout 0.05 \
  --aux_weight 0.15 \
  --aux_start_epoch 0 \
  --aux_warmup_epochs 0 \
  --skeleton_iterations 8 \
  --road_crop_probability 0.0 \
  --road_occlusion_probability 0.0 \
  --progressive_unfreeze \
  --unfreeze_dual_branch_epoch 0 \
  --unfreeze_layer3_epoch 5 \
  --unfreeze_layer2_epoch 10 \
  --unfreeze_all_epoch 15 \
  --freeze_encoder_bn \
  --num_workers 4 \
  --prefetch_factor 2 \
  --use_amp \
  --channels_last \
  --pretrained_checkpoint /kaggle/working/best_fixed_iou.pt \
  --transfer_weights ema \
  --save_dir ./checkpoints/abl_no_dappm \
  --ablate_dappm


In [ ]:
!python test_native.py \
  --ckpt ./checkpoints/abl_no_dappm/best_calibrated_road_iou.pt \
  --dataset massachusetts --subset test178 --thr 0.58 --weights ema \
  --tta-mode flip4 --tta-merge probabilities --amp float16


In [ ]:
%cd /kaggle/working/road-extraction

!OMP_NUM_THREADS=2 TORCH_NCCL_ASYNC_ERROR_HANDLING=1 \
torchrun --standalone --nproc_per_node=2 \
  train.py \
  --dataset massachusetts \
  --data_root /kaggle/input/datasets/datnguyentien204/massachu/massachusets \
  --train_image_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/images \
  --train_mask_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/labels \
  --train_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/train.txt \
  --test_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/test.txt \
  --crop_size 1024 \
  --batch_size 8 \
  --accumulation_steps 1 \
  --epochs 100 \
  --lr 5e-5 \
  --dual_branch_lr_factor 0.75 \
  --backbone_lr_factor 0.20 \
  --early_encoder_lr_factor 0.10 \
  --weight_decay 1e-4 \
  --warmup_epochs 3 \
  --warmup_start_factor 0.10 \
  --min_lr_ratio 0.0 \
  --grad_clip 5.0 \
  --ema_decay 0.999 \
  --detail_channels 96 \
  --semantic_channels 192 \
  --dappm_channels 32 \
  --dappm_pool_sizes 1 2 4 8 \
  --detail_blocks 2 2 \
  --semantic_blocks 2 \
  --fusion_blocks 1 \
  --bilateral_fusion spatial \
  --decoder_s4_channels 64 \
  --decoder_s2_channels 32 \
  --full_channels 24 \
  --dropout 0.05 \
  --aux_weight 0.15 \
  --aux_start_epoch 0 \
  --aux_warmup_epochs 0 \
  --skeleton_iterations 8 \
  --road_crop_probability 0.0 \
  --road_occlusion_probability 0.0 \
  --progressive_unfreeze \
  --unfreeze_dual_branch_epoch 0 \
  --unfreeze_layer3_epoch 5 \
  --unfreeze_layer2_epoch 10 \
  --unfreeze_all_epoch 15 \
  --freeze_encoder_bn \
  --num_workers 4 \
  --prefetch_factor 2 \
  --use_amp \
  --channels_last \
  --pretrained_checkpoint /kaggle/working/best_fixed_iou.pt \
  --transfer_weights ema \
  --save_dir ./checkpoints/abl_no_detail_refine --ablate_detail_refinement \
  --ablate_dappm


In [ ]:
!python test_native.py \
  --ckpt ./checkpoints/abl_no_detail_refine/best_calibrated_road_iou.pt \
  --dataset massachusetts --subset test178 --thr 0.58 --weights ema \
  --tta-mode flip4 --tta-merge probabilities --amp float16


In [ ]:
%cd /kaggle/working/road-extraction

!OMP_NUM_THREADS=2 TORCH_NCCL_ASYNC_ERROR_HANDLING=1 \
torchrun --standalone --nproc_per_node=2 \
  train.py \
  --dataset massachusetts \
  --data_root /kaggle/input/datasets/datnguyentien204/massachu/massachusets \
  --train_image_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/images \
  --train_mask_dir /kaggle/input/datasets/datnguyentien204/massachu/massachusets/labels \
  --train_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/train.txt \
  --test_list /kaggle/input/datasets/datnguyentien204/massachu/massachusets/test.txt \
  --crop_size 1024 \
  --batch_size 8 \
  --accumulation_steps 1 \
  --epochs 100 \
  --lr 5e-5 \
  --dual_branch_lr_factor 0.75 \
  --backbone_lr_factor 0.20 \
  --early_encoder_lr_factor 0.10 \
  --weight_decay 1e-4 \
  --warmup_epochs 3 \
  --warmup_start_factor 0.10 \
  --min_lr_ratio 0.0 \
  --grad_clip 5.0 \
  --ema_decay 0.999 \
  --detail_channels 96 \
  --semantic_channels 192 \
  --dappm_channels 32 \
  --dappm_pool_sizes 1 2 4 8 \
  --detail_blocks 2 2 \
  --semantic_blocks 2 \
  --fusion_blocks 1 \
  --bilateral_fusion spatial \
  --decoder_s4_channels 64 \
  --decoder_s2_channels 32 \
  --full_channels 24 \
  --dropout 0.05 \
  --aux_weight 0.15 \
  --aux_start_epoch 0 \
  --aux_warmup_epochs 0 \
  --skeleton_iterations 8 \
  --road_crop_probability 0.0 \
  --road_occlusion_probability 0.0 \
  --progressive_unfreeze \
  --unfreeze_dual_branch_epoch 0 \
  --unfreeze_layer3_epoch 5 \
  --unfreeze_layer2_epoch 10 \
  --unfreeze_all_epoch 15 \
  --freeze_encoder_bn \
  --num_workers 4 \
  --prefetch_factor 2 \
  --use_amp \
  --channels_last \
  --pretrained_checkpoint /kaggle/working/best_fixed_iou.pt \
  --transfer_weights ema \
  --ablate_dappm \
  --save_dir ./checkpoints/abl_no_strip_pooling --ablate_strip_pooling

In [ ]:
!python test_native.py \
  --ckpt ./checkpoints/abl_no_strip_pooling/best_calibrated_road_iou.pt \
  --dataset massachusetts --subset test178 --thr 0.58 --weights ema \
  --tta-mode flip4 --tta-merge probabilities --amp float16


In [ ]:
# from pathlib import Path
# import zipfile
# import torch

# # Thư mục .pt đã bị Kaggle giải nén
# src = Path(
#     "/kaggle/input/datasets/datnguyentien204/road-8-27-2026/best_calibrated_road_iou.pt"
# )

# # File weight được khôi phục
# dst = Path("/kaggle/working/best_fixed_iou.pt")

# assert src.is_dir(), f"Không tìm thấy thư mục đã giải nén: {src}"

# # Tìm thư mục chứa data.pkl của checkpoint PyTorch
# data_pkls = list(src.rglob("data.pkl"))
# assert len(data_pkls) == 1, f"Tìm thấy {len(data_pkls)} file data.pkl"

# checkpoint_root = data_pkls[0].parent

# # Khôi phục đúng cấu trúc ZIP serialization của PyTorch
# with zipfile.ZipFile(
#     dst,
#     mode="w",
#     compression=zipfile.ZIP_STORED,
#     allowZip64=True,
# ) as archive:
#     for file in checkpoint_root.rglob("*"):
#         if file.is_file():
#             relative = file.relative_to(checkpoint_root)
#             archive.write(file, arcname=str(Path("archive") / relative))

# print("Đã tạo:", dst)
# print(f"Dung lượng: {dst.stat().st_size / 1024**2:.1f} MB")

# # Kiểm tra file weight
# checkpoint = torch.load(dst, map_location="cpu", weights_only=False)
# print("Load thành công")
# print("Các key:", checkpoint.keys())

In [ ]:
# from pathlib import Path
# import zipfile
# import torch

# # Thư mục .pt đã bị Kaggle giải nén
# src = Path(
#     "/kaggle/input/datasets/datnguyentien204/2343345/best_calibrated_road_iou.pt"
# )

# # File weight được khôi phục
# dst = Path("/kaggle/working/best_fixed_iou.pt")

# assert src.is_dir(), f"Không tìm thấy thư mục đã giải nén: {src}"

# # Tìm thư mục chứa data.pkl của checkpoint PyTorch
# data_pkls = list(src.rglob("data.pkl"))
# assert len(data_pkls) == 1, f"Tìm thấy {len(data_pkls)} file data.pkl"

# checkpoint_root = data_pkls[0].parent

# # Khôi phục đúng cấu trúc ZIP serialization của PyTorch
# with zipfile.ZipFile(
#     dst,
#     mode="w",
#     compression=zipfile.ZIP_STORED,
#     allowZip64=True,
# ) as archive:
#     for file in checkpoint_root.rglob("*"):
#         if file.is_file():
#             relative = file.relative_to(checkpoint_root)
#             archive.write(file, arcname=str(Path("archive") / relative))

# print("Đã tạo:", dst)
# print(f"Dung lượng: {dst.stat().st_size / 1024**2:.1f} MB")

# # Kiểm tra file weight
# checkpoint = torch.load(dst, map_location="cpu", weights_only=False)
# print("Load thành công")
# print("Các key:", checkpoint.keys())

In [ ]:
# !torchrun --standalone --nproc_per_node=2 train.py \
#   --dataset massachusetts \
#   --pretrained_checkpoint /kaggle/working/best_fixed_iou.pt \
#   --transfer_weights ema \
#   --no-progressive_unfreeze \
#   --detail_channels 96 --semantic_channels 192 --dappm_channels 32 \
#   --dappm_pool_sizes 1 2 4 8 --detail_blocks 2 2 \
#   --bilateral_fusion spatial \
#   --decoder_s4_channels 64 --decoder_s2_channels 32 --full_channels 24 \
#   --crop_size 1024 --batch_size 8 --accumulation_steps 1 \
#   --lr 2e-4 \
#   --dual_branch_lr_factor 0.75 --backbone_lr_factor 0.20 --early_encoder_lr_factor 0.10 \
#   --warmup_epochs 1 \
#   --fixed_road_weight 2.0 \
#   --epochs 20 \
#   --save_dir ./checkpoints/dual_branch_roadnet/lr_probe_2e-4 \
#   --seed 42

In [ ]:
# from pathlib import Path
# import zipfile
# import torch

# # Thư mục .pt đã bị Kaggle giải nén
# src = Path(
#     "/kaggle/input/datasets/datnguyentien204/road-8-27-2026/best_calibrated_road_iou.pt"
# )

# # File weight được khôi phục
# dst = Path("/kaggle/working/best_fixed_iou1.pt")

# assert src.is_dir(), f"Không tìm thấy thư mục đã giải nén: {src}"

# # Tìm thư mục chứa data.pkl của checkpoint PyTorch
# data_pkls = list(src.rglob("data.pkl"))
# assert len(data_pkls) == 1, f"Tìm thấy {len(data_pkls)} file data.pkl"

# checkpoint_root = data_pkls[0].parent

# # Khôi phục đúng cấu trúc ZIP serialization của PyTorch
# with zipfile.ZipFile(
#     dst,
#     mode="w",
#     compression=zipfile.ZIP_STORED,
#     allowZip64=True,
# ) as archive:
#     for file in checkpoint_root.rglob("*"):
#         if file.is_file():
#             relative = file.relative_to(checkpoint_root)
#             archive.write(file, arcname=str(Path("archive") / relative))

# print("Đã tạo:", dst)
# print(f"Dung lượng: {dst.stat().st_size / 1024**2:.1f} MB")

# # Kiểm tra file weight
# checkpoint = torch.load(dst, map_location="cpu", weights_only=False)
# print("Load thành công")
# print("Các key:", checkpoint.keys())

In [ ]:
# %cd /kaggle/working/road-extraction

# !python /kaggle/working/road-extraction/compare_reparameterization.py \
#   --ckpt /kaggle/working/best_fixed_iou.pt \
#   --weights ema \
#   --form deploy \
#   --height 1024 \
#   --width 1024 \
#   --memory-repeats 5 \
#   --data-parallel \
#   --json-out /kaggle/working/weaving_table5_resources.json

In [ ]:
%cd /kaggle/working/road-extraction

!python test_native.py \
  --ckpt ./checkpoints/roadfusion_spatial_bilateral_ft40/best_calibrated_road_iou.pt \
  --dataset massachusetts \
  --subset test178 \
  --thr 0.58 \
  --weights ema \
  --tta-mode flip4 \
  --tta-merge probabilities \
  --amp float16

In [ ]:
# %cd /kaggle/working/road-extraction

# !python test_native.py \
#   --ckpt /kaggle/working/best_fixed_iou.pt \
#   --dataset deepglobe \
#   --subset deepglobe_test \
#   --thr 0.58 \
#   --weights ema \
#   --tta-mode flip4 \
#   --tta-merge probabilities \
#   --amp float16

In [ ]:
# %cd /kaggle/working/road-extraction

# !python test_native.py \
#   --ckpt /kaggle/working/best_fixed_iou.pt \
#   --dataset deepglobe \
#   --weights ema \
#   --subset deepglobe_test \
#   --thr 0.50 \
#   --tta-mode flip4 \
#   --amp auto \
#   --tile-batch-size 2 \
#   --out ./checkpoints/deepglobe_imagenet_fulltrain_lr2e4_100ep/val300_probs.npz

In [ ]:
# !python compare_reparameterization.py \
#     --ckpt /kaggle/working/best_fixed_iou.pt \
#     --weights ema \
#     --amp float16 \
#     --no-channels-last \
#     --warmup 20 \
#     --repeats 100 \
#     --skip-eval